# Size Comparison: Qualitative SVG Samples (Tiny → XL)

**Purpose**: Generate samples from each µP model size for Section 5.6 qualitative comparison.

| Step | What | Est. Time |
|------|------|----------|
| 1 | Setup (Drive + deps) | 1 min |
| 2 | Tokenizer verification | <1 min |
| 3 | Generate: 5 sizes × (10 uncond + 15 prefix) | 5-10 min |
| 4 | Render + Stats + Grid | 1 min |

**Runtime**: A100 GPU (Colab Pro). Total ~10 min.

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/svg-scaling-project
!git pull
!pip install -r requirements.txt -q

In [ ]:
import torch, subprocess, os, json, hashlib, shutil, sys
from pathlib import Path

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify all 5 checkpoints exist
sizes = ['tiny', 'small', 'medium', 'large', 'xl']
for size in sizes:
    ckpt = f'results/runs/mup/{size}/best_model.pt'
    assert os.path.exists(ckpt), f'Missing checkpoint: {ckpt}'
    mb = os.path.getsize(ckpt) / 1024 / 1024
    print(f'  {size:>8s}: {ckpt} ({mb:.1f} MB)')

print('\n✓ All 5 checkpoints found')

---
## 2. Tokenizer Verification

The models were trained with the Day 6 tokenizer (vocab 166).
If the local tokenizer is the old Day 3 version, rebuild it.

In [ ]:
DAY3_TOKENIZER_MD5 = 'e7b88618354c5dc3920a2427f79b42d7'

with open('tokenizer/bpe_svg.json', 'rb') as f:
    tok_hash = hashlib.md5(f.read()).hexdigest()

if tok_hash == DAY3_TOKENIZER_MD5:
    print('Tokenizer is Day 3 version → rebuilding...')
    subprocess.run([
        'python', 'src/tokenize_data.py',
        '--input-dir', 'data/processed',
        '--output-dir', 'data/tokenized',
        '--vocab-size', '4096',
        '--max-token-len', '0',
    ], check=True)
    print('Tokenizer rebuilt.')
else:
    print(f'Tokenizer OK (hash: {tok_hash[:12]}...)')

from tokenizers import Tokenizer
tok = Tokenizer.from_file('tokenizer/bpe_svg.json')
print(f'Vocab size: {tok.get_vocab_size()}')

---
## 3. Generate Samples

Per size: 10 unconditional + 5 per prefix × 3 prefixes = 25 samples.

Settings: `temperature=0.8`, `top_k=40`, `max_new_tokens=4096`.

In [ ]:
import time

OUTPUT_BASE = 'results/samples_size_comparison'

# Clean previous run
if os.path.exists(OUTPUT_BASE):
    shutil.rmtree(OUTPUT_BASE)

prefixes = {
    'single_shape_group': open('results/prefixes_v2/single_shape_group.svg').read(),
    'face_partial': open('results/prefixes_v2/face_partial.svg').read(),
    'open_path': open('results/prefixes_v2/open_path.svg').read(),
}

TEMP = '0.8'
TOP_K = '40'
MAX_TOKENS = '4096'
N_UNCOND = '10'
N_PREFIX = '5'

t_total = time.time()

for size in sizes:
    config = f'configs/{size}.yaml'
    ckpt = f'results/runs/mup/{size}/best_model.pt'
    t0 = time.time()
    print(f'\n{"="*60}')
    print(f'  {size.upper()}')
    print(f'{"="*60}')

    # Unconditional
    uncond_dir = f'{OUTPUT_BASE}/{size}/unconditional'
    print(f'  Unconditional ({N_UNCOND} samples)...')
    subprocess.run([
        'python', 'src/generate.py',
        '--config', config,
        '--checkpoint', ckpt,
        '--tokenizer', 'tokenizer/bpe_svg.json',
        '--num-samples', N_UNCOND,
        '--temperature', TEMP,
        '--top-k', TOP_K,
        '--max-tokens', MAX_TOKENS,
        '--output-dir', uncond_dir,
        '--mup',
        '--device', 'cuda',
    ], check=True, capture_output=True)

    n_svg = len(list(Path(uncond_dir).glob('*.svg')))
    n_inc = len(list(Path(uncond_dir).glob('*_incomplete*')))
    print(f'    → {n_svg} complete, {n_inc} incomplete')

    # Prefix-conditioned
    for pname, prefix_text in prefixes.items():
        prefix_dir = f'{OUTPUT_BASE}/{size}/prefix/{pname}'
        print(f'  Prefix: {pname} ({N_PREFIX} samples)...')
        subprocess.run([
            'python', 'src/generate.py',
            '--config', config,
            '--checkpoint', ckpt,
            '--tokenizer', 'tokenizer/bpe_svg.json',
            '--num-samples', N_PREFIX,
            '--prefix', prefix_text,
            '--temperature', TEMP,
            '--top-k', TOP_K,
            '--max-tokens', MAX_TOKENS,
            '--output-dir', prefix_dir,
            '--mup',
            '--device', 'cuda',
        ], check=True, capture_output=True)

        n_svg = len(list(Path(prefix_dir).glob('*.svg')))
        n_inc = len(list(Path(prefix_dir).glob('*_incomplete*')))
        print(f'    → {n_svg} complete, {n_inc} incomplete')

    elapsed = time.time() - t0
    print(f'  {size} done in {elapsed:.0f}s')

total_elapsed = time.time() - t_total
print(f'\n{"="*60}')
print(f'All generation done in {total_elapsed/60:.1f} min')
print(f'{"="*60}')

---
## 4. Render + Stats + Grid Image

In [ ]:
import cairosvg
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw, ImageFont
import io

OUTPUT_BASE = 'results/samples_size_comparison'

stats = {}

for size in sizes:
    size_dir = Path(OUTPUT_BASE) / size
    svg_files = sorted(size_dir.rglob('*.svg'))
    incomplete_files = sorted(size_dir.rglob('*_incomplete.txt'))
    total = len(svg_files) + len(incomplete_files)

    xml_valid = 0
    render_ok = 0

    for svg_path in svg_files:
        # XML validity
        try:
            ET.parse(svg_path)
            xml_valid += 1
        except ET.ParseError:
            continue

        # Render to PNG
        png_path = svg_path.with_suffix('.png')
        try:
            cairosvg.svg2png(
                url=str(svg_path),
                write_to=str(png_path),
                output_width=200,
                output_height=200,
            )
            render_ok += 1
        except Exception as e:
            pass

    stats[size] = {
        'total': total,
        'complete': len(svg_files),
        'incomplete': len(incomplete_files),
        'xml_valid': xml_valid,
        'render_success': render_ok,
        'completion_rate': len(svg_files) / total if total else 0,
        'xml_validity_rate': xml_valid / total if total else 0,
        'render_rate': render_ok / total if total else 0,
    }
    print(f'{size:>8s}: {len(svg_files):>2d}/{total} complete, '
          f'{xml_valid} xml-valid, {render_ok} rendered')

# Save stats
with open(f'{OUTPUT_BASE}/size_comparison_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(f'\nStats saved to {OUTPUT_BASE}/size_comparison_stats.json')

In [ ]:
# Create grid image: 5 rows (sizes) × 5 cols (unconditional samples)

cell_size = 200
cols = 5
rows = len(sizes)
pad = 2
label_h = 30

grid_w = cols * (cell_size + pad) + pad
grid_h = rows * (cell_size + pad + label_h) + pad
grid = Image.new('RGB', (grid_w, grid_h), 'white')
draw = ImageDraw.Draw(grid)

try:
    font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 14)
except (OSError, IOError):
    font = ImageFont.load_default()

for row, size in enumerate(sizes):
    uncond_dir = Path(OUTPUT_BASE) / size / 'unconditional'
    pngs = sorted(uncond_dir.glob('*.png'))[:cols]
    y_off = row * (cell_size + pad + label_h) + pad

    # Label
    summary_path = Path(f'results/runs/mup/{size}/summary.json')
    label = size.upper()
    if summary_path.exists():
        with open(summary_path) as f:
            s = json.load(f)
        label += f'  ({s["n_params"]/1e6:.1f}M, loss={s["best_val_loss"]:.3f})'
    draw.text((pad + 4, y_off + 2), label, fill='black', font=font)

    for col, png_path in enumerate(pngs):
        x = col * (cell_size + pad) + pad
        y = y_off + label_h
        try:
            img = Image.open(png_path).resize((cell_size, cell_size))
            grid.paste(img, (x, y))
        except Exception:
            pass

grid_path = f'{OUTPUT_BASE}/size_comparison_grid.png'
grid.save(grid_path)
print(f'Grid saved to {grid_path}')

# Display in notebook
from IPython.display import display
display(grid)

In [ ]:
# Also create a prefix-conditioned grid: 5 sizes × 3 prefixes (best sample each)

prefix_names = ['single_shape_group', 'face_partial', 'open_path']
p_cols = len(prefix_names)
p_rows = len(sizes)

pgrid_w = p_cols * (cell_size + pad) + pad
pgrid_h = p_rows * (cell_size + pad + label_h) + pad + label_h  # extra row for prefix labels
pgrid = Image.new('RGB', (pgrid_w, pgrid_h), 'white')
pdraw = ImageDraw.Draw(pgrid)

# Column headers
for col, pname in enumerate(prefix_names):
    x = col * (cell_size + pad) + pad + 4
    pdraw.text((x, 2), pname.replace('_', ' '), fill='gray', font=font)

for row, size in enumerate(sizes):
    y_off = row * (cell_size + pad + label_h) + pad + label_h

    # Row label
    label = size.upper()
    pdraw.text((pad + 4, y_off + 2), label, fill='black', font=font)

    for col, pname in enumerate(prefix_names):
        prefix_dir = Path(OUTPUT_BASE) / size / 'prefix' / pname
        pngs = sorted(prefix_dir.glob('*.png'))[:1]
        if pngs:
            x = col * (cell_size + pad) + pad
            y = y_off + label_h
            try:
                img = Image.open(pngs[0]).resize((cell_size, cell_size))
                pgrid.paste(img, (x, y))
            except Exception:
                pass

pgrid_path = f'{OUTPUT_BASE}/prefix_comparison_grid.png'
pgrid.save(pgrid_path)
print(f'Prefix grid saved to {pgrid_path}')
display(pgrid)

In [ ]:
# Summary table
print(f'{"Size":>8s}  {"Complete":>8s}  {"XML Valid":>9s}  {"Rendered":>8s}  {"Comp%":>6s}  {"Render%":>7s}')
print('-' * 55)
for size in sizes:
    s = stats[size]
    print(f'{size:>8s}  {s["complete"]:>3d}/{s["total"]:<4d}  '
          f'{s["xml_valid"]:>4d}/{s["total"]:<4d}  '
          f'{s["render_success"]:>3d}/{s["total"]:<4d}  '
          f'{s["completion_rate"]*100:>5.1f}%  '
          f'{s["render_rate"]*100:>6.1f}%')

print(f'\n✓ Results in {OUTPUT_BASE}/')
print('  - size_comparison_grid.png (Figure 10: unconditional)')
print('  - prefix_comparison_grid.png (prefix-conditioned)')
print('  - size_comparison_stats.json')